In [0]:
# First, ensure we have the latest databricks-sdk with Lakebase support
%pip install --upgrade 'databricks-sdk>=0.89.0' --quiet

In [0]:
# Restart Python to load the updated databricks-sdk
dbutils.library.restartPython()

In [0]:
# Discover Lakebase project and endpoint path using WorkspaceClient
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Based on your connection UI:
# - Project: my-transactional-db (discovered)
# - Branch: production
# - Compute/Endpoint: primary

project_name = "my-transactional-db"
branch_name = "production"
endpoint_name = "primary"

# Construct the full endpoint path
endpoint_path = f"projects/{project_name}/branches/{branch_name}/endpoints/{endpoint_name}"

print(f"✓ Constructed endpoint path: {endpoint_path}")
print(f"\nThis will be used to generate the OAuth token for Lakebase connection.")

✓ Constructed endpoint path: projects/my-transactional-db/branches/production/endpoints/primary

This will be used to generate the OAuth token for Lakebase connection.


In [0]:
query = f"SELECT * FROM public.notebook_execution_log WHERE notebook_name = '{notebook_name}'"

df = spark.read.jdbc(
    url=jdbc_url,
    table=f"({query}) AS filtered_log",
    properties=properties
)

if not df.isEmpty():
    last_completed_timestamp = df.collect()[0][2]

print(last_completed_timestamp)

2026-06-13 04:36:25.837196


In [0]:
from databricks.sdk import WorkspaceClient

# Initialize WorkspaceClient (uses notebook context authentication)
w = WorkspaceClient()

# Lakebase connection details from your UI:
# - Project: my-transactional-db
# - Branch: production
# - Compute: primary
# - Database: databricks_postgres
# - Role: username@company.com

endpoint_path = "projects/my-transactional-db/branches/production/endpoints/primary"

print(f"Generating OAuth token for endpoint: {endpoint_path}")

# Generate OAuth token for database connection (1-hour expiration)
credential = w.postgres.generate_database_credential(
    endpoint=endpoint_path
)

print(f"✓ Successfully generated Lakebase OAuth token")
print(f"Token: {credential.token[:50]}...")
print(f"Expires: {credential.expire_time}")

# Use the generated token to connect to Lakebase via JDBC
jdbc_url = "jdbc:postgresql://ep-gentle-resonance-d8avqn7u.database.us-east-2.cloud.databricks.com/databricks_postgres?sslmode=require"

properties = {
    "user": "username@company.com",
    "password": credential.token,
    "driver": "org.postgresql.Driver"
}


Generating OAuth token for endpoint: projects/my-transactional-db/branches/production/endpoints/primary
✓ Successfully generated Lakebase OAuth token
Token: eyJraWQiOiI2NDZiZWZkNGY5NjYwMTdiNjk1MjRjOTRlMjcxNz...
Expires: seconds: 1781328122



id,notebook_name,last_completed_timestamp


In [0]:
from pyspark.sql.functions import current_timestamp, lit

notebook_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

df_notebook_info = spark.createDataFrame(
    [(notebook_name,)],
    ["notebook_name"]
).withColumn("last_completed_timestamp", current_timestamp())

# Use individual options for serverless compatibility
df_notebook_info.write.format("postgresql").mode("append").options(
    host="ep-gentle-resonance-d8avqn7u.database.us-east-2.cloud.databricks.com",
    port="5432",
    database="databricks_postgres",
    dbtable="public.notebook_execution_log",
    user="username@company.com",
    password=credential.token
).save()